## Some resources:
1. 2019 LLVM Developers’ Meeting: A. Warzynski “Writing an LLVM Pass: 101” - https://www.youtube.com/watch?v=ar7cJl2aBuU
2. LLVM docs - Writing an LLVM Pass  https://llvm.org/docs/WritingAnLLVMNewPMPass.html
3. IR transformation example from LLVM: https://github.com/llvm/llvm-project/blob/main/llvm/examples/IRTransforms/SimplifyCFG.cpp
4. https://bholt.org/posts/llvm-quick-tricks.html
5. https://www.cs.cornell.edu/courses/cs6120/2025fa/lesson/7//#tasks
6. Excellent resource for passes: https://github.com/banach-space/llvm-tutor/tree/main

## Some notes:
- An analysis pass gathers information about the IR. It inspects the IR to compute some reusable information. For example:
    - Dominance relationships (DominatorTreeAnalysis)
    - Loop nesting structure (LoopAnalysis)
    - Alias sets (AAResultsAnalysis)
    - Dataflow facts like reaching definitions
These results are cached in the FunctionAnalysisManager or ModuleAnalysisManager, so other passes can query them efficiently.Its key traits
    - Does not modify IR.
    - Can be reused by many transformations.
    - Must define a Result type and a run() method that returns it.
    - Other passes can request it with FAM.getResult<YourAnalysis>(F)
- All LLVM passes inherit from the CRTP mix-in PassInfoMixin<PassT>. The pass should have a run() method which returns a PreservedAnalyses and takes in some unit of IR along with an analysis manager. For example, a function pass would have a PreservedAnalyses run(Function &F, FunctionAnalysisManager &FAM); method.
- Examine control-flow edges using predecessors() and successors() from CFG.h, and traverse instructions. For each instruction we can collect opcode histograms using I.getOpcodeName(). For function calls, we use dyn_cast<CallBase> to get the CalledFunction and accumulate call frequency.
- Dominance defines control dependencies: a block A dominates B if every path to B passes through A. LLVM builds a DominatorTree per function using standard algorithms (Lengauer-Tarjan). The LoopInfo analysis then identifies natural loops (via back-edges in the DomTree) and their nesting depth.We use LoopAnalysis from the FunctionAnalysisManager.
    - LoopInfo &LI = AM.getResult<LoopAnalysis>(F);
      LoopInfo is derived from the dominator tree: a back-edge from a block to one of its dominators forms a natural loop. Note: Loop nesting depth informs scheduling, tiling, and unrolling strategies.

### Real world use:
- Estimate data reuse, memory footprint, or loop trip counts.
- Build dominator or dependence information for hardware mapping.
- Custom hardwares benefit from loop flattening or software pipelining guided by this analysis.

## Typical workflow:

- Set up environment variables so that clang, opt, and llvm-config are found.
- Write the pass source code into a .cpp file.
- Compile the pass into a .dylib.
- Create a test C file, lower it to LLVM IR (.ll).
- Run opt with your pass and capture the output.

## This work:

It performs static control-flow and data-flow analysis building a DominatorTree per
function, enumerating successors to form the CFG, querying LoopInfo for loop nesting, 
and generating opcode histograms. I also integrated call-graph detection by visiting
CallBase instructions. The pass runs standalone via the new plugin API and demonstrates
a full analysis pipeline from basic blocks to loops.

In [2]:
import sys, os

LLVM_HOME = "/opt/homebrew/Cellar/llvm/21.1.5"

os.environ["PATH"] = f"{LLVM_HOME}/bin:" + os.environ["PATH"]
os.environ["LDFLAGS"] = f"-L{LLVM_HOME}/lib"
os.environ["CPPFLAGS"] = f"-I{LLVM_HOME}/include"

!which clang
!which clang++
!which opt
!which llvm-config
!clang++ --version
!llvm-config --version

!echo 'export PATH="/opt/homebrew/opt/llvm/bin:$PATH"' >> ~/.zshrc
!echo 'export LDFLAGS="-L/opt/homebrew/opt/llvm/lib"' >> ~/.zshrc
!echo 'export CPPFLAGS="-I/opt/homebrew/opt/llvm/include"' >> ~/.zshrc
!source ~/.zshrc

/opt/homebrew/Cellar/llvm/21.1.5/bin/clang
/opt/homebrew/Cellar/llvm/21.1.5/bin/clang++
/opt/homebrew/Cellar/llvm/21.1.5/bin/opt
/opt/homebrew/Cellar/llvm/21.1.5/bin/llvm-config
Homebrew clang version 21.1.5
Target: arm64-apple-darwin24.6.0
Thread model: posix
InstalledDir: /opt/homebrew/Cellar/llvm/21.1.5/bin
Configuration file: /opt/homebrew/etc/clang/arm64-apple-darwin24.cfg
21.1.5


In [3]:
pass_code = r"""
#include "llvm/IR/Function.h"
#include "llvm/IR/PassManager.h"
#include "llvm/Passes/PassBuilder.h"
#include "llvm/Passes/PassPlugin.h"
#include "llvm/Support/raw_ostream.h"
#include "llvm/IR/Instructions.h"
#include "llvm/IR/InstrTypes.h"
#include "llvm/IR/CFG.h"
#include "llvm/Analysis/LoopInfo.h"
#include "llvm/IR/Dominators.h"

#include <map>
#include <string>


using namespace llvm;

struct FuncAnalysisPass : public PassInfoMixin<FuncAnalysisPass> {
  PreservedAnalyses run(Function &F, FunctionAnalysisManager &FAM) {
    // Print to stderr so we definitely see it
    errs() << "\n[FuncAnalysisPass] on function: " << F.getName() << "\n";
    
    DominatorTree DT(F);
    DT.recalculate(F);
    
    LoopInfo &LI = FAM.getResult<LoopAnalysis>(F);
    
    unsigned BBcount = 0, Instcount= 0, loads=0, stores=0;
    std::map<std::string, unsigned> opcodeCounts;
    std::map<std::string, unsigned> callCounts;

    // --- 1. Count BBs, instructions, opcode breakdown, loads/stores, calls ---
    for (auto &BB : F) {
      ++BBcount;
      Instcount += BB.size();
      const auto *Node = DT.getNode(&BB);
      if (Node && Node->getIDom())
          errs() << "  Immediate Dominator: " << Node->getIDom()->getBlock()->getName() << "\n";
      else
          errs() << "  Immediate Dominator: (entry block)\n";

      errs() << "  Dominates: ";
      for (auto *Child : *Node) {
          if (!Child->getBlock()->hasName())
              Child->getBlock()->setName("bb_dom_" + std::to_string(BBcount++));
          errs() << Child->getBlock()->getName() << " ";
      }
      errs() << "\n";

      
      for (auto &I : BB) {
            opcodeCounts[I.getOpcodeName()]++;
    
            if (isa<LoadInst>(&I)) loads++;
            if (isa<StoreInst>(&I)) stores++;
    
            if (auto *callInst = dyn_cast<CallBase>(&I)) {
                if (Function *calledFunc = callInst->getCalledFunction()) {
                    callCounts[calledFunc->getName().str()]++;
                    errs() << "    Calls function: " << calledFunc->getName() << "\n";
                }
            }
      }
    }

    // 2. Loop Detection using LoopAnalysis
    
    outs() << "Loops in function: " << std::distance(LI.begin(), LI.end()) << "\n";
    unsigned loopCount = 0;
    for (auto *L : LI) {
        BasicBlock *Header = L->getHeader();
        if (!Header->hasName())
            Header->setName("loop_header_" + std::to_string(loopCount));
        errs() << "  Loop " << loopCount
           << " header: " << Header->getName()
           << " (depth=" << L->getLoopDepth() << ")\n";
        loopCount++;
    }


    // 3. Print CFG edges
    for (auto &BB : F) {
        if (!BB.hasName())
            BB.setName("bb_" + F.getName() + "_" + std::to_string(BBcount++));

        errs() << "  BasicBlock " << BB.getName() << " successors: ";
        const Instruction *T = BB.getTerminator();
        for (unsigned i = 0; i < T->getNumSuccessors(); ++i) {
            BasicBlock *Succ = T->getSuccessor(i);
            if (!Succ->hasName())
                Succ->setName("succ_" + std::to_string(i));
            errs() << Succ->getName() << " ";
        }
        errs() << "\n";
    }

    errs() << "  Function: " << F.getName()
           << " | BasicBlocks: " << BBcount
           << " | Instructions: " << Instcount
           << " | Load Instructions: " << loads
           << " | Store Instructions: " << stores 
           << " | Loops: " << loopCount << "\n";

    // --- 5. Instruction breakdown ---
    errs() << "  Instruction breakdown:\n";
        for (auto &entry : opcodeCounts) {
            outs() << "    " << entry.first << ": " << entry.second << "\n";
        }

    // --- 6. Call counts ---
        errs() << "  Calls:\n";
        for (auto &entry : callCounts)
            errs() << "    " << entry.first << ": " << entry.second << "\n";

    errs().flush();
    return PreservedAnalyses::all();
  }
   // Run even if functions have optnone 
   //A required pass is a pass that may not be skipped. An example of a required pass
   //is AlwaysInlinerPass, which must always be run to preserve alwaysinline semantics. 
   //Pass managers are required since they may contain other required passes. An example
   //of how a pass can be skipped is the optnone function attribute, which specifies that
   //optimizations should not be run on the function. Required passes will still be run on
   //optnone functions.


  static bool isRequired() { return true; }
};


// Register plugin
extern "C" LLVM_ATTRIBUTE_WEAK PassPluginLibraryInfo llvmGetPassPluginInfo() {
  // This line confirms the .dylib was loaded.
  errs() << "FuncAnalysisPass plugin loaded!\n";

  return {
    LLVM_PLUGIN_API_VERSION, "FuncAnalysisPass", "v0.1",
    [](PassBuilder &PB) {
      // 1) Allow `-passes="func-analysis"` at *module* level by adapting to function pass
      PB.registerPipelineParsingCallback(
        [](StringRef Name, ModulePassManager &MPM,
           ArrayRef<PassBuilder::PipelineElement>) {
          if (Name == "func-analysis") {
            errs() << "FuncAnalysisPass added to module pipeline\n";
            MPM.addPass(createModuleToFunctionPassAdaptor(FuncAnalysisPass()));
            return true;
          }
          return false;
        });

      // 2) Also allow `-passes="function(func-analysis)"`
      PB.registerPipelineParsingCallback(
        [](StringRef Name, FunctionPassManager &FPM,
           ArrayRef<PassBuilder::PipelineElement>) {
          if (Name == "func-analysis") {
            errs() << "FuncAnalysisPass added to function pipeline\n";
            FPM.addPass(FuncAnalysisPass());
            return true;
          }
          return false;
        });
    }
  };
}
"""

with open("FuncAnalysisPass.cpp", "w") as f:
    f.write(pass_code)


In [4]:
c_code = r"""
#include <stdio.h>

int add(int a, int b) {
    return a + b;
}

int sum(int n) {
    int s = 0;
    for (int i = 0; i < n; i++)
        s += i;
    return s;
}

int foo(int x) {
  if (x > 0)
    return x + 1;
  else
    return x + 2;
}

int main() {
    int x = add(1, 2);
    int y = sum(x);
    int z = foo(2); 
    printf("%d\n", x);
    printf("%d\n", y);
    printf("%d\n", z);
    return 0;
}
"""

with open("add.c", "w") as f:
    f.write(c_code)

In [5]:
!clang -emit-llvm -S -O0 add.c -o add.ll

In [6]:
!/opt/homebrew/opt/llvm/bin/clang++ -std=c++20 -fPIC -shared \
  "$(/opt/homebrew/opt/llvm/bin/llvm-config --cxxflags --ldflags)" \
  -I/opt/homebrew/opt/llvm/include \
  -Wl,-undefined,dynamic_lookup \
  FuncAnalysisPass.cpp -o FuncAnalysisPass.dylib


In [7]:
!/opt/homebrew/opt/llvm/bin/opt \
  -load-pass-plugin=./FuncAnalysisPass.dylib \
  -passes="function(func-analysis)" \
  -disable-output add.ll

FuncAnalysisPass plugin loaded!
FuncAnalysisPass added to function pipeline

[FuncAnalysisPass] on function: add
  Immediate Dominator: (entry block)
  Dominates: 
Loops in function: 0
  BasicBlock bb_add_1 successors: 
  Function: add | BasicBlocks: 2 | Instructions: 8 | Load Instructions: 2 | Store Instructions: 2 | Loops: 0
  Instruction breakdown:
    add: 1
    alloca: 2
    load: 2
    ret: 1
    store: 2
  Calls:

[FuncAnalysisPass] on function: sum
  Immediate Dominator: (entry block)
  Dominates: bb_dom_1 
  Immediate Dominator: 
  Dominates: bb_dom_3 bb_dom_4 
  Immediate Dominator: bb_dom_1
  Dominates: bb_dom_6 
  Immediate Dominator: bb_dom_3
  Dominates: 
  Immediate Dominator: bb_dom_1
  Dominates: 
Loops in function: 1
  Loop 0 header: bb_dom_1 (depth=1)
  BasicBlock bb_sum_9 successors: bb_dom_1 
  BasicBlock bb_dom_1 successors: bb_dom_3 bb_dom_4 
  BasicBlock bb_dom_3 successors: bb_dom_6 
  BasicBlock bb_dom_6 successors: bb_dom_1 
  BasicBlock bb_dom_4 successors: 

Function: add
Two basic blocks (entry and return).
Two loads/stores from stack variables (alloca/load/store).
No loops or branches (as expected).

Function: sum
One loop detected, with a clear loop header.
The CFG successors correctly show the backedge (succ_01 -> loop_header_0).
The dominator tree prints every block’s relationship, confirming control-flow consistency.

Function: foo
Conditional branch with merge point (succ_01).
Dominator info shows one entry dominating both branches.

Function: main
Calls to all subroutines correctly identified.
Proper call instruction counting and symbol resolution.
Demonstrates we can extract call graph edges.

### Some more notes:

- Can you explain how LoopInfo and DominatorTree are related?
    - LoopInfo is built on top of the dominator tree : a natural loop exists when a block has a 
backedge to one of its dominators. The dominator tree identifies headers, and LoopInfo
walks those relationships to form loop nests

- Importance of LoopInfo:
LoopInfo discovers loop nests and identifies headers, exits, and backedges in the CFG.
It powers loop unrolling, vectorization, LICM (loop-invariant code motion), and induction-variable analysis.

- Importance of Dominance:
Dominance forms the backbone of SSA construction, optimization scoping, and control-flow analysis.

- How would you detect hot or high-power regions in code? 
    - By combining opcode statistics from my FunctionPass with loop nesting from LoopInfo 
and trip-count estimates from SCEV, I can approximate operation density — a proxy for 
compute or power hotspots

- Each Instruction in LLVM IR has an opcode and can be dynamically queried (isa<>, dyn_cast<>).
We classify instructions by opcode to estimate memory access intensity — loads/stores often dominate power cost.